# Notebook 14 - Audio Understanding, Classification, and Tagging

This notebook moves beyond speech transcription into sound classification, audio-text retrieval, and zero-shot labeling.


## What you will learn

- how CLAP-style embeddings support audio-text retrieval
- when audio understanding needs more than a transcript
- how to build zero-shot audio labels and confidence tables


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib librosa soundfile torchaudio
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "14_audio_understanding_classification_and_tagging"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Simulate zero-shot audio labels

Zero-shot labels are a practical bridge into audio understanding when you do not have a supervised classifier for every sound.


In [ ]:
audio_labels = pd.DataFrame([
    {"candidate_label": "glass breaking", "score": 0.71},
    {"candidate_label": "keyboard typing", "score": 0.14},
    {"candidate_label": "door closing", "score": 0.09},
    {"candidate_label": "speech", "score": 0.06},
]).sort_values("score", ascending=False)
audio_labels


## Step 2: Store paired audio-text embeddings

Contrastive models make audio retrieval and tagging feel more like text or image retrieval.


In [ ]:
audio_embedding = np.array([0.82, 0.11, 0.07])
text_embeddings = {
    "glass breaking": np.array([0.79, 0.14, 0.07]),
    "speech": np.array([0.18, 0.72, 0.10]),
    "keyboard typing": np.array([0.33, 0.21, 0.46]),
}
rows = []
for label, vec in text_embeddings.items():
    score = float(audio_embedding @ vec)
    rows.append({"label": label, "similarity": round(score, 3)})
pd.DataFrame(rows).sort_values("similarity", ascending=False)


## Step 3: Design an audio tag schema

Typed outputs are especially helpful once audio systems must trigger downstream actions.


In [ ]:
audio_tag_record = {
    "clip_id": "incident-audio-01",
    "top_label": "glass breaking",
    "confidence": 0.71,
    "secondary_labels": ["alarm", "speech"],
    "evidence_window": "00:03-00:05",
}
print(json.dumps(audio_tag_record, indent=2))


## Exercises and extensions

1. Compare a transcript-first approach against an audio-labeling approach on a non-speech clip.
1. Build a small retrieval table that matches text descriptions to audio clips using embedding similarity.
